# Spatial Predictive Maintenance of Water Pumps — XGBoost + geopandas + pyproj

โน้ตบุ๊กนี้แปลงมาจาก `code_03_pipeline_xgboost_geopandas.py` เพื่อให้รันทีละ
เซลล์ เห็นผลลัพธ์ระหว่างทางได้ทันที ดีบักง่ายกว่าการรันทั้งไฟล์ในครั้งเดียว

**ก่อนรัน:** อัปโหลด `train_values.csv`, `train_labels.csv`, `test_values.csv`
ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้ (หรือแก้ `DATA_DIR` ในเซลล์ตั้งค่าด้านล่าง)

## ติดตั้งไลบรารีที่จำเป็น (รันครั้งเดียวตอนเปิดใหม่)

In [ ]:
!pip install -q xgboost geopandas shapely pyproj


## นำเข้าไลบรารี (imports)

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    fbeta_score,
    make_scorer,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline

import geopandas as gpd
from pyproj import CRS
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
%matplotlib inline


## ตั้งค่าพาธข้อมูล/ผลลัพธ์

In [ ]:
RANDOM_STATE = 42

# แก้ 2 พาธนี้ให้ตรงกับที่คุณเก็บไฟล์ไว้ (ค่าเริ่มต้น = โฟลเดอร์ปัจจุบัน)
DATA_DIR = Path(".")
OUT_DIR = Path("./outputs_xgboost")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. โหลดข้อมูล + รวมไฟล์

`train_values.csv` (รายละเอียดปั๊ม) กับ `train_labels.csv` (คำตอบ) แยกกันอยู่
คนละไฟล์ ต้อง `.merge(on="id")` เอามาต่อกันก่อน

In [ ]:
def load_data():
    values = pd.read_csv(DATA_DIR / "train_values.csv")
    labels = pd.read_csv(DATA_DIR / "train_labels.csv")
    test_values = pd.read_csv(DATA_DIR / "test_values.csv")
    df = values.merge(labels, on="id")
    return df, test_values

df, test_values = load_data()
df.head()


## 2. EDA (สำรวจข้อมูลก่อนลงมือ)

ฟังก์ชันนี้แค่ **นับ** ปัญหาที่มีในข้อมูล (ยังไม่แก้อะไร) — พิกัดผิดปกติ, สัดส่วน
ของแต่ละ class ในคำตอบ ฯลฯ

In [ ]:
TZ_BOUNDS = dict(lon_min=29.0, lon_max=41.0, lat_min=-12.0, lat_max=-0.9)


def run_eda(df: pd.DataFrame):
    report_lines = []

    def log(s=""):
        print(s)
        report_lines.append(s)

    log("=" * 70)
    log("EDA SUMMARY")
    log("=" * 70)
    log(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
    log("\nTarget distribution (status_group):")
    log(df["status_group"].value_counts().to_string())

    invalid_coords = (df.longitude == 0) | (df.longitude < TZ_BOUNDS["lon_min"]) | \
                      (df.longitude > TZ_BOUNDS["lon_max"]) | \
                      (df.latitude < TZ_BOUNDS["lat_min"]) | (df.latitude > TZ_BOUNDS["lat_max"])
    log(f"\nInvalid / out-of-Tanzania coordinates: {invalid_coords.sum():,} "
        f"({invalid_coords.mean()*100:.1f}% of rows)")

    (OUT_DIR / "00_eda_report.txt").write_text("\n".join(report_lines))
    return invalid_coords


invalid_mask = run_eda(df)


## 3. ทำความสะอาดข้อมูล (Cleaning)

- `gps_height`/`population`/`construction_year` ที่เป็น 0 คือค่า **หายไป**
  ไม่ใช่ 0 จริง ๆ → แปลงเป็น `NaN`
- `public_meeting`/`permit` เป็น boolean ในข้อมูลดิบ — ต้องแปลงเป็น string
  ก่อน เพราะ XGBoost ไม่รับ category ที่เป็น boolean (bug ที่เจอจากการรันจริง)
- พิกัดที่ผิดปกติ (นอกกรอบแทนซาเนีย) → ตั้งเป็น `NaN` ไว้ก่อน เดี๋ยวไปซ่อมใน
  ขั้นตอนถัดไป

In [ ]:
def clean_common(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date_recorded"] = pd.to_datetime(df["date_recorded"])
    df["construction_year"] = df["construction_year"].replace(0, np.nan)
    df["age"] = df["date_recorded"].dt.year - df["construction_year"]
    df.loc[(df["age"] < 0) | (df["age"] > 60), "age"] = np.nan
    df["gps_height"] = df["gps_height"].where(df["gps_height"] != 0, np.nan)
    df["population"] = df["population"].where(df["population"] != 0, np.nan)

    # [FIX] public_meeting/permit เป็น boolean -> XGBoost ไม่รับ category ที่เป็น
    # boolean ต้องแปลงเป็น string ก่อน (NaN ยังเป็น NaN เหมือนเดิม)
    for c in ["public_meeting", "permit"]:
        df[c] = df[c].map({True: "True", False: "False"})

    return df


def clean_geo(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    invalid = (
        (df.longitude == 0)
        | (df.longitude < TZ_BOUNDS["lon_min"]) | (df.longitude > TZ_BOUNDS["lon_max"])
        | (df.latitude < TZ_BOUNDS["lat_min"]) | (df.latitude > TZ_BOUNDS["lat_max"])
    )
    df["invalid_coords"] = invalid.astype(int)
    df.loc[invalid, ["longitude", "latitude"]] = np.nan
    for c in ["subvillage", "ward", "lga", "scheme_name", "funder", "installer"]:
        df[c] = df[c].astype(str).str.strip().str.lower().replace({"nan": np.nan})
    return df


df = clean_common(df)
df = clean_geo(df)
df[["gps_height", "population", "age", "public_meeting", "permit", "invalid_coords"]].head()


## 4. สร้าง GIS Feature

### ทำไมต้อง "project" พิกัด

lat/lon องศาปกติวัดระยะทางตรง ๆ ไม่แม่นยำ (เพราะโลกกลม ไม่ใช่กระดาษแบน)
ต้องแปลง (project) เป็นหน่วยเมตรบนระนาบแบนก่อน ถึงจะคำนวณระยะทาง/จัดกลุ่ม
ตามระยะทางได้ถูกต้อง — เลือกใช้ **Azimuthal Equidistant (AEQD)** ที่อิงจุด
ศูนย์กลางของแทนซาเนีย แทนการบังคับใช้ UTM โซนเดียว เพราะแทนซาเนียครอบคลุม
ทั้ง 2 โซน UTM (36S/37S) ถ้าเลือกโซนใดโซนหนึ่งอีกฝั่งของประเทศจะบิดเบือน

In [ ]:
WGS84 = "EPSG:4326"  # ระบบพิกัดมาตรฐานของ GPS

TZ_PROJECTED_CRS = CRS.from_proj4(
    "+proj=aeqd +lat_0=-6.5 +lon_0=35.0 +datum=WGS84 +units=m +no_defs"
)


def project_xy(lat, lon):
    """แปลง lat/lon (องศา) เป็นพิกัดหน่วยเมตร ด้วย geopandas/pyproj จริง"""
    gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(lon, lat), crs=WGS84)
    gdf_proj = gdf.to_crs(TZ_PROJECTED_CRS)
    return gdf_proj.geometry.x.values, gdf_proj.geometry.y.values


def validate_against_shapefile(df, shapefile_path, lon_col="longitude", lat_col="latitude"):
    """ตัวเลือกเสริม: เช็คพิกัดแบบ point-in-polygon กับ shapefile ขอบเขตจริง
    (เช่นจาก GADM) แทนการใช้กรอบสี่เหลี่ยมคร่าว ๆ ข้ามไปเฉย ๆ ถ้าไม่มีไฟล์"""
    if not Path(shapefile_path).exists():
        print(f"[validate_against_shapefile] ไม่พบไฟล์ {shapefile_path} — ข้ามขั้นตอนนี้")
        return pd.Series(False, index=df.index)
    boundary = gpd.read_file(shapefile_path).to_crs(WGS84)
    points = gpd.GeoDataFrame(
        df.copy(), geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs=WGS84,
    )
    joined = gpd.sjoin(points, boundary, how="left", predicate="within")
    outside = joined["index_right"].isna()
    print(f"[validate_against_shapefile] จุดที่อยู่นอกขอบเขตจริง: {outside.sum():,} / {len(df):,}")
    return outside


### ระยะทางถึงเมืองใหญ่ (Haversine formula)

สูตรมาตรฐานสำหรับคำนวณระยะทางเส้นตรงบนผิวทรงกลม ระหว่าง 2 จุดที่รู้ lat/lon

In [ ]:
MAJOR_CITIES = {
    "Dar es Salaam": (-6.7924, 39.2083), "Dodoma": (-6.1630, 35.7516),
    "Mwanza": (-2.5164, 32.9175), "Arusha": (-3.3869, 36.6830),
    "Mbeya": (-8.9094, 33.4608), "Morogoro": (-6.8235, 37.6822),
    "Tanga": (-5.0692, 39.0962), "Kigoma": (-4.8766, 29.6266),
    "Zanzibar City": (-6.1659, 39.2026), "Songea": (-10.6833, 35.6500),
    "Iringa": (-7.7694, 35.6919), "Musoma": (-1.5017, 33.8010),
    "Shinyanga": (-3.6614, 33.4237), "Singida": (-4.8180, 34.7500),
    "Sumbawanga": (-7.9667, 31.6167),
}


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


### `FreqEncoder` — เข้ารหัสคอลัมน์ที่มีชื่อไม่ซ้ำเยอะมาก (funder/installer)

แปลงแต่ละชื่อเป็น "ความถี่ที่ชื่อนี้ปรากฏในข้อมูล train" แทนการใส่เป็น category
ตรง ๆ (`fit` เรียนรู้ความถี่จาก train เท่านั้น — สำคัญมากเพื่อกัน data leakage
ตอนทำ cross-validation)

In [ ]:
class FreqEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        self.maps_ = {c: X[c].value_counts(normalize=True) for c in self.cols}
        return self

    def transform(self, X):
        X = X.copy()
        for c, m in self.maps_.items():
            X[f"{c}_freq"] = X[c].map(m).fillna(0.0)
        return X


### `GeoFeaturizer` — หัวใจของการสร้าง feature เชิงพื้นที่

ทำ 4 อย่าง: (1) ซ่อมพิกัดที่หายไปด้วย median ของภูมิภาค (2) จัดกลุ่มปั๊มตาม
ตำแหน่งด้วย **KMeans** (3) คำนวณระยะทาง/ความหนาแน่นด้วย **KD-tree** (4) นับ
ความถี่ชื่อ — สถิติทุกตัว fit จาก training fold เท่านั้นเสมอ

In [ ]:
class GeoFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=40, density_radius_km=5.0):
        self.n_clusters = n_clusters
        self.density_radius_km = density_radius_km

    def fit(self, X, y=None):
        X = X.copy()
        self.region_coord_medians_ = (
            X.loc[X.invalid_coords == 0].groupby("region")[["latitude", "longitude"]].median()
        )
        self.global_coord_median_ = X.loc[X.invalid_coords == 0][["latitude", "longitude"]].median()

        X_imputed = self._impute_coords(X)

        xy = np.column_stack(project_xy(X_imputed.latitude.values, X_imputed.longitude.values))
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, n_init=10, random_state=RANDOM_STATE)
        self.kmeans_.fit(xy)

        self.kdtree_ = cKDTree(xy)

        self.freq_maps_ = {}
        for c in ["subvillage", "ward", "lga", "scheme_name", "funder", "installer"]:
            self.freq_maps_[c] = X[c].value_counts(normalize=True)

        return self

    def _impute_coords(self, X):
        X = X.copy()
        missing = X["latitude"].isna()
        if missing.any():
            reg_med = X.loc[missing, "region"].map(self.region_coord_medians_["latitude"])
            X.loc[missing, "latitude"] = reg_med.fillna(self.global_coord_median_["latitude"])
            reg_med_lon = X.loc[missing, "region"].map(self.region_coord_medians_["longitude"])
            X.loc[missing, "longitude"] = reg_med_lon.fillna(self.global_coord_median_["longitude"])
        return X

    def transform(self, X):
        X = self._impute_coords(X.copy())
        xy = np.column_stack(project_xy(X.latitude.values, X.longitude.values))

        X["spatial_cluster"] = self.kmeans_.predict(xy)

        dists = [haversine_km(X.latitude.values, X.longitude.values, lat_c, lon_c)
                 for lat_c, lon_c in MAJOR_CITIES.values()]
        X["dist_nearest_city_km"] = np.min(np.column_stack(dists), axis=1)

        radius_m = self.density_radius_km * 1000
        X["local_density"] = self.kdtree_.query_ball_point(xy, r=radius_m, return_length=True)

        nn_dist, _ = self.kdtree_.query(xy, k=2)
        X["nearest_pump_m"] = np.where(nn_dist[:, 0] < 1e-6, nn_dist[:, 1], nn_dist[:, 0])

        for c, freq_map in self.freq_maps_.items():
            X[f"{c}_freq"] = X[c].map(freq_map).fillna(0.0)

        return X


## 5. เลือกชุด Feature ที่จะใช้เข้าโมเดล

`BASELINE_*` = 18 feature ของ repo ต้นฉบับ (ไม่มีตำแหน่งเลย)
`GIS_*` = baseline + feature ตำแหน่งทั้งหมด — แยก 2 ชุดไว้เทียบกันตรง ๆ

In [ ]:
BASELINE_NUM = ["amount_tsh", "gps_height", "population", "age",
                 "funder_freq", "installer_freq"]
BASELINE_CAT = ["basin", "region", "public_meeting",
                 "scheme_management", "permit", "extraction_type_class",
                 "management_group", "payment_type", "quality_group",
                 "quantity_group", "source_class", "waterpoint_type_group"]

GIS_NUM = BASELINE_NUM + ["latitude", "longitude", "region_code", "district_code",
                            "dist_nearest_city_km", "local_density", "nearest_pump_m",
                            "subvillage_freq", "ward_freq", "lga_freq",
                            "scheme_name_freq"]
GIS_CAT = BASELINE_CAT + ["spatial_cluster", "invalid_coords"]


def fit_category_levels(df, cat_cols):
    """[FIX] จำ categories จาก training fold ไว้ กันรหัส category เพี้ยนระหว่าง fold"""
    return {c: sorted(df[c].dropna().unique().tolist()) for c in cat_cols}


def to_model_frame(df, num_cols, cat_cols, cat_levels):
    X = df[num_cols + cat_cols].copy()
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    for c in cat_cols:
        X[c] = pd.Categorical(X[c], categories=cat_levels[c])
    return X


## 6. เทรนโมเดล + วัดผล

### F2-score: ทำไมไม่ใช้ accuracy เฉย ๆ
งานบำรุงรักษาเชิงป้องกัน พลาดปั๊มที่กำลังจะเสีย (false negative) อันตรายกว่า
ส่งช่างไปดูปั๊มที่ยังดีอยู่ (false positive) — F2 ให้น้ำหนัก recall มากกว่า
precision 2 เท่า, `average="macro"` เฉลี่ยแบบให้ทุก class สำคัญเท่ากัน

### ทำไมต้องเข้ารหัส label เป็นตัวเลข
XGBoost (ตั้งแต่ 1.3.2) ต้องการ `y` เป็นจำนวนเต็ม 0..n-1 เท่านั้น ไม่รับ string
เช่น `"functional"` ตรง ๆ (bug ที่เจอจากการรันจริง) ต้อง `LabelEncoder` ก่อน
แล้วแปลงกลับเป็น string ตอนออกรายงานผล

In [ ]:
def f2_macro(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2, average="macro")


F2_SCORER = make_scorer(f2_macro)


def encode_labels(y):
    le = LabelEncoder()
    y_enc = pd.Series(le.fit_transform(y), index=y.index, name=y.name)
    return y_enc, le


def make_xgb():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.08,
        max_depth=6,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


### Builder classes — ห่อทุกขั้นตอนเป็นชิ้นเดียวที่ใช้กับ `Pipeline`/`cross_val_score` ได้

แยก `.fit()`/`.transform()` ตามมาตรฐาน sklearn เพื่อให้ `cross_val_score`
เรียก `.fit()` เฉพาะกับ training fold ของแต่ละรอบ CV ให้อัตโนมัติ (กัน data
leakage โดยไม่ต้องเขียน loop เอง)

In [ ]:
class GeoFrameBuilder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.geo_ = GeoFeaturizer()

    def fit(self, X, y=None):
        self.geo_.fit(X, y)
        X_transformed = self.geo_.transform(X)
        self.cat_levels_ = fit_category_levels(X_transformed, GIS_CAT)
        return self

    def transform(self, X):
        X2 = self.geo_.transform(X)
        return to_model_frame(X2, GIS_NUM, GIS_CAT, self.cat_levels_)


class BaselineFrameBuilder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_ = FreqEncoder(["funder", "installer"])

    def fit(self, X, y=None):
        self.freq_.fit(X, y)
        X_transformed = self.freq_.transform(X)
        self.cat_levels_ = fit_category_levels(X_transformed, BASELINE_CAT)
        return self

    def transform(self, X):
        X2 = self.freq_.transform(X)
        return to_model_frame(X2, BASELINE_NUM, BASELINE_CAT, self.cat_levels_)


def evaluate(df_clean, label, builder, y):
    pipe = Pipeline([("features", builder), ("model", make_xgb())])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, df_clean, y, cv=skf, scoring=F2_SCORER, n_jobs=1)
    print(f"[{label}] F2-macro per fold: {np.round(scores, 4)}")
    print(f"[{label}] F2-macro mean={scores.mean():.4f}  std={scores.std():.4f}")
    return scores


## รันจริง: เตรียม label

In [ ]:
y = df["status_group"]
y_enc, label_encoder = encode_labels(y)
print("classes:", list(label_encoder.classes_))


## รันจริง: โมเดล baseline (ไม่มี geo) — 5-fold CV

In [ ]:
base_scores = evaluate(df, "baseline", BaselineFrameBuilder(), y_enc)


## รันจริง: โมเดล gis_enhanced (มี geo ครบ) — 5-fold CV

In [ ]:
gis_scores = evaluate(df, "gis_enhanced", GeoFrameBuilder(), y_enc)


## เปรียบเทียบคะแนน 2 โมเดล

In [ ]:
results = pd.DataFrame({
    "model": ["baseline (18 features, no geo)", "gis_enhanced (+ geo features)"],
    "f2_macro_mean": [base_scores.mean(), gis_scores.mean()],
    "f2_macro_std": [base_scores.std(), gis_scores.std()],
})
results.to_csv(OUT_DIR / "04_model_comparison_xgb.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(results["model"], results["f2_macro_mean"], yerr=results["f2_macro_std"],
       color=["#457b9d", "#2a9d8f"], capsize=6)
ax.set_ylabel("F2-macro (5-fold CV)")
ax.set_title("Baseline vs GIS-enhanced (XGBoost)")
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "05_model_comparison_xgb.png", dpi=140)
plt.show()

results


## Classification report + Confusion matrix

ใช้ `cross_val_predict` เพื่อให้ทุกแถวได้ "คำทำนายตอนที่มันอยู่ใน validation
fold" — ภาพสะท้อนสิ่งที่จะเกิดขึ้นจริงตอน deploy ไม่ใช่การทำนายข้อมูลที่โมเดล
เคยเห็นตอนเทรนไปแล้ว

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
gis_pipe = Pipeline([("features", GeoFrameBuilder()), ("model", make_xgb())])
y_pred_enc = cross_val_predict(gis_pipe, df, y_enc, cv=skf, n_jobs=1)
y_pred = label_encoder.inverse_transform(y_pred_enc)

report = classification_report(y, y_pred)
(OUT_DIR / "06_classification_report_xgb.txt").write_text(report)
print(report)

labels_order = ["functional", "functional needs repair", "non functional"]
cm = confusion_matrix(y, y_pred, labels=labels_order)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order).plot(
    ax=ax, cmap="Blues", colorbar=False, xticks_rotation=20)
ax.set_title("XGBoost GIS-enhanced model — out-of-fold confusion matrix")
plt.tight_layout()
plt.savefig(OUT_DIR / "07_confusion_matrix_xgb.png", dpi=140)
plt.show()


## Feature importance — feature ไหนสำคัญที่สุด

In [ ]:
builder = GeoFrameBuilder().fit(df, y_enc)
X_full = builder.transform(df)
model = make_xgb()
model.fit(X_full, y_enc)

imp_df = pd.DataFrame({
    "feature": X_full.columns,
    "importance_gain": model.feature_importances_,
}).sort_values("importance_gain", ascending=False)
imp_df.to_csv(OUT_DIR / "08_feature_importance_xgb.csv", index=False)

top = imp_df.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(top["feature"], top["importance_gain"], color="#2a9d8f")
ax.set_xlabel("XGBoost feature importance (gain)")
ax.set_title("Top 20 features — XGBoost GIS-enhanced model")
plt.tight_layout()
plt.savefig(OUT_DIR / "09_feature_importance_xgb.png", dpi=140)
plt.show()

imp_df.head(10)


## บันทึกข้อมูลที่ clean + สร้าง feature ครบแล้ว

In [ ]:
full_clean = builder.geo_.transform(df)
cols_to_save = ["id"] + GIS_NUM + GIS_CAT + ["status_group"]
full_clean[cols_to_save].to_csv(OUT_DIR / "cleaned_train_data_gis_xgb.csv", index=False)
print("Done. Outputs written to", OUT_DIR)


## สรุป

- `baseline`: F2-macro จาก 5-fold CV โดยไม่มีข้อมูลตำแหน่งเลย
- `gis_enhanced`: baseline + feature ตำแหน่งทั้งหมด (spatial_cluster,
  ระยะทาง, ความหนาแน่น, lat/lon, region_code/district_code, ฯลฯ)

ถ้าอยากลองต่อ: ไฟล์ `code_04_pruned_gis_experiment_xgboost.py` ทดสอบว่า
"คัดเฉพาะ feature ตำแหน่งที่สำคัญจริง" (7 ตัว) แทนที่จะใส่ทุกตัว จะให้ผลดีกว่า
`gis_enhanced` ที่ใส่ครบทุกตัวหรือไม่ — ในเวอร์ชัน sklearn พบว่าใส่ครบทุกตัว
กลับแย่กว่า baseline เสียอีก แต่คัดเฉพาะตัวที่สำคัญแล้วชนะทั้งคู่